In [1]:
import os, cv2, random, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [2]:
DATA_ROOT = "/kaggle/input/competitions/dlp-26-t-2-week-11-assignment/competition-data/competition-data"

TRAIN_IMG_DIR   = f"{DATA_ROOT}/training/images"
TRAIN_DEPTH_DIR = f"{DATA_ROOT}/training/depths"
VAL_IMG_DIR     = f"{DATA_ROOT}/validation/images"
VAL_DEPTH_DIR   = f"{DATA_ROOT}/validation/depths"
TEST_IMG_DIR    = f"{DATA_ROOT}/testing/images"

OUT_DIR = "/kaggle/working"
PRED_VAL_DIR  = f"{OUT_DIR}/val_preds"
PRED_TEST_DIR = f"{OUT_DIR}/test_preds"
os.makedirs(PRED_VAL_DIR, exist_ok=True)
os.makedirs(PRED_TEST_DIR, exist_ok=True)

IMG_SIZE   = 224      # train at higher res for quality, downsample later
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 3e-4
NUM_WORKERS = 4

In [3]:
train_imgs = sorted(os.listdir(TRAIN_IMG_DIR))
train_depths = sorted(os.listdir(TRAIN_DEPTH_DIR))
val_imgs = sorted(os.listdir(VAL_IMG_DIR))
test_imgs = sorted(os.listdir(TEST_IMG_DIR))

print(len(train_imgs), len(train_depths), len(val_imgs), len(test_imgs))
print(train_imgs[:3])

sample = cv2.imread(os.path.join(TRAIN_IMG_DIR, train_imgs[0]))
sample_d = cv2.imread(os.path.join(TRAIN_DEPTH_DIR, train_depths[0]), cv2.IMREAD_UNCHANGED)
print("RGB shape:", sample.shape, "Depth shape:", sample_d.shape, sample_d.dtype)

6686 6686 836 836
['10012003.png', '10012004.png', '10022005.png']
RGB shape: (256, 256, 3) Depth shape: (256, 256) uint8


In [4]:
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

class DepthDataset(Dataset):
    def __init__(self, img_dir, depth_dir=None, filenames=None, transform=None):
        self.img_dir = img_dir
        self.depth_dir = depth_dir
        self.filenames = filenames or sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.depth_dir is not None:
            depth = cv2.imread(os.path.join(self.depth_dir, fname), cv2.IMREAD_UNCHANGED)
            if depth.ndim == 3:
                depth = cv2.cvtColor(depth, cv2.COLOR_BGR2GRAY)
            depth = cv2.resize(depth, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            depth = depth.astype(np.float32) / 255.0

            augmented = self.transform(image=img)
            img_t = augmented["image"]
            depth_t = torch.from_numpy(depth).unsqueeze(0)
            return img_t, depth_t, fname
        else:
            augmented = self.transform(image=img)
            return augmented["image"], fname

train_ds = DepthDataset(TRAIN_IMG_DIR, TRAIN_DEPTH_DIR, train_imgs, train_tf)
val_ds   = DepthDataset(VAL_IMG_DIR, VAL_DEPTH_DIR, val_imgs, eval_tf)
test_ds  = DepthDataset(TEST_IMG_DIR, None, test_imgs, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

/tmp/ipykernel_24/1162523883.py:5: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_c + skip_c, out_c)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)   # /2,  64ch
        self.pool = resnet.maxpool                                         # /4
        self.enc1 = resnet.layer1   # /4,  64ch
        self.enc2 = resnet.layer2   # /8,  128ch
        self.enc3 = resnet.layer3   # /16, 256ch
        self.enc4 = resnet.layer4   # /32, 512ch

        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 32)

        self.final_up = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.head = nn.Sequential(
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x0 = self.stem(x)          # /2
        x1 = self.pool(x0)
        x1 = self.enc1(x1)         # /4
        x2 = self.enc2(x1)         # /8
        x3 = self.enc3(x2)         # /16
        x4 = self.enc4(x3)         # /32

        d4 = self.up4(x4, x3)
        d3 = self.up3(d4, x2)
        d2 = self.up2(d3, x1)
        d1 = self.up1(d2, x0)
        out = self.final_up(d1)
        out = self.head(out)
        return out

model = ResNetUNet(pretrained=True).to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 195MB/s]


In [6]:
def rmse_loss(pred, target):
    return torch.sqrt(F.mse_loss(pred, target) + 1e-8)

def gradient_loss(pred, target):
    def grad(x):
        gx = x[:, :, :, 1:] - x[:, :, :, :-1]
        gy = x[:, :, 1:, :] - x[:, :, :-1, :]
        return gx, gy
    pgx, pgy = grad(pred)
    tgx, tgy = grad(target)
    return (pgx - tgx).abs().mean() + (pgy - tgy).abs().mean()

def normalized_rmse_loss(pred, target):
    # per-sample min-max normalize both pred and target (differentiable),
    # mimicking the eval pipeline's per-image contrast stretch
    def norm(x):
        b = x.shape[0]
        x_flat = x.view(b, -1)
        mn = x_flat.min(dim=1, keepdim=True)[0]
        mx = x_flat.max(dim=1, keepdim=True)[0]
        x_norm = (x_flat - mn) / (mx - mn + 1e-6)
        return x_norm.view_as(x)
    p, t = norm(pred), norm(target)
    return torch.sqrt(F.mse_loss(p, t) + 1e-8)

def combined_loss(pred, target):
    return (
        0.5 * rmse_loss(pred, target)
        + 0.3 * normalized_rmse_loss(pred, target)
        + 0.2 * gradient_loss(pred, target)
    )

#def combined_loss(pred, target):
#    return rmse_loss(pred, target) + 0.2 * gradient_loss(pred, target)

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

best_val_rmse = float("inf")
BEST_PATH = f"{OUT_DIR}/best_model.pth"

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_rmse, n = 0.0, 0.0, 0
    for imgs, depths, _ in loader:
        imgs, depths = imgs.to(DEVICE, non_blocking=True), depths.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast():
                preds = model(imgs)
                loss = combined_loss(preds, depths)
                rmse = rmse_loss(preds, depths)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_rmse += rmse.item() * bs
        n += bs
    return total_loss / n, total_rmse / n

for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_rmse = run_epoch(train_loader, train=True)
    val_loss, val_rmse = run_epoch(val_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | train_rmse={train_rmse:.4f} | val_rmse={val_rmse:.4f} | {time.time()-t0:.1f}s")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  -> saved best model (val_rmse={best_val_rmse:.4f})")

gc.collect(); torch.cuda.empty_cache()

/tmp/ipykernel_24/2341085511.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_24/2341085511.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/30 | train_rmse=0.1742 | val_rmse=0.1184 | 52.2s
  -> saved best model (val_rmse=0.1184)
Epoch 2/30 | train_rmse=0.1145 | val_rmse=0.1015 | 33.6s
  -> saved best model (val_rmse=0.1015)
Epoch 3/30 | train_rmse=0.1091 | val_rmse=0.0970 | 35.4s
  -> saved best model (val_rmse=0.0970)
Epoch 4/30 | train_rmse=0.1056 | val_rmse=0.1012 | 34.4s
Epoch 5/30 | train_rmse=0.1025 | val_rmse=0.0893 | 34.4s
  -> saved best model (val_rmse=0.0893)
Epoch 6/30 | train_rmse=0.0989 | val_rmse=0.0872 | 34.7s
  -> saved best model (val_rmse=0.0872)
Epoch 7/30 | train_rmse=0.0972 | val_rmse=0.0867 | 34.4s
  -> saved best model (val_rmse=0.0867)
Epoch 8/30 | train_rmse=0.0953 | val_rmse=0.0828 | 34.4s
  -> saved best model (val_rmse=0.0828)
Epoch 9/30 | train_rmse=0.0934 | val_rmse=0.0807 | 34.6s
  -> saved best model (val_rmse=0.0807)
Epoch 10/30 | train_rmse=0.0924 | val_rmse=0.0795 | 34.5s
  -> saved best model (val_rmse=0.0795)
Epoch 11/30 | train_rmse=0.0891 | val_rmse=0.0780 | 34.5s
  -> saved 

In [8]:
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.eval()

def predict_and_save(loader, out_dir, orig_dir):
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                imgs, _, fnames = batch
            else:
                imgs, fnames = batch
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast():
                preds = model(imgs)  # B,1,H,W in [0,1]
            preds = preds.squeeze(1).cpu().numpy()

            for pred, fname in zip(preds, fnames):
                # resize to match original image dims for saving (optional),
                # imgs2csv will resize to 128x128 anyway so 128x128 output is fine directly
                pred_img = (pred * 255.0).clip(0, 255).astype(np.uint8)
                pred_img = cv2.resize(pred_img, (128, 128), interpolation=cv2.INTER_LINEAR)
                cv2.imwrite(os.path.join(out_dir, fname), pred_img)

predict_and_save(test_loader, PRED_TEST_DIR, TEST_IMG_DIR)
print("Saved test predictions:", len(os.listdir(PRED_TEST_DIR)))

/tmp/ipykernel_24/1664987231.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Saved test predictions: 836


In [9]:
def images_to_csv_with_metadata(image_folder, output_csv):
    data = []
    for idx, filename in enumerate(sorted(os.listdir(image_folder))):
        if filename.endswith(".png"):
            filepath = os.path.join(image_folder, filename)
            image = cv2.imread(filepath, cv2.IMREAD_UNCHANGED)
            image = cv2.resize(image, (128, 128))
            image = image / 255.
            image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-6)
            image = np.uint8(image * 255.)
            image_flat = image.flatten()
            row = [idx, filename] + image_flat.tolist()
            data.append(row)

    num_columns = len(data[0]) - 2 if data else 0
    column_names = ["id", "ImageID"] + [i for i in range(num_columns)]
    df = pd.DataFrame(data, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f"Saved {output_csv} with {len(df)} rows")

images_to_csv_with_metadata(PRED_TEST_DIR, f"{OUT_DIR}/submission.csv")

Saved /kaggle/working/submission.csv with 836 rows
